# Сдвиг во времени

In [ ]:
'''
Больше обучающих примеров из той же истории. Для каждого клиента (train и test) дата оценки сдвигается на 4 и 8 месяцев назад,
признаки считаются по заказам до сдвинутого отсечения, таргет — был ли заказ в следующие 3 месяца (целиком в видимой истории).
Ловушка отбора: в выборку попали только клиенты с заказом в последние 365 дней до настоящего отсечения; наивный сдвиг завышает
таргет (0.39 и 0.46 против 0.34). Поэтому берём псевдоклиентов с давностью не больше 365 − длина сдвига.
Признаки — набор v5 (485). Главная проверка — перенос на март (обучение на январе и феврале): он дважды совпал с Public
(раунд 2: март +0.017 → Public +0.015; раунд 3: март +0.0009 → Public +0.0009). Шум оцениваем парным бутстрепом по клиентам марта.
CPU, около часа. Выход: submission_shift_lgb.csv (одна LightGBM на настоящих срезах + псевдосрезах) и shift.json.
'''
import itertools
import json
import os
import re
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display
from scipy.special import ndtr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 5
EARLY_STOP = 200
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print("GPU:", GPU)
T0 = time.time()


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

# псевдосрезы: та же история, дата оценки сдвинута на SHIFTS месяцев назад.
# В выборку попадают только клиенты с заказом не дальше 365 дней до настоящего отсечения. Чтобы этот отбор не завышал
# таргет псевдосреза, берём только тех, у кого давность на сдвинутую дату не больше 365 − длина сдвига в днях.
SHIFTS = [4, 8]
ID_OFF = 1_000_000
real = cust.copy()
aug_c, aug_o = [], []
for k, s in enumerate(SHIFTS, 1):
    c = real[["customer_id", "scoring_date", "cutoff"]].rename(columns={"cutoff": "real_cutoff"})
    c["scoring_date"] = c["scoring_date"] - pd.DateOffset(months=s)
    c["cutoff"] = c["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
    c["win_end"] = c["scoring_date"] + pd.DateOffset(months=TARGET_MONTHS)
    assert (c["win_end"] <= c["real_cutoff"]).all()            # целевое окно псевдосреза целиком в видимой истории
    m = orders[["customer_id", "order_date"]].merge(c, on="customer_id")
    hist = m[m["order_date"] <= m["cutoff"]]
    rec = (hist["cutoff"] - hist["order_date"]).dt.days.groupby(hist["customer_id"]).min()
    c["rec"] = c["customer_id"].map(rec)
    c = c[c["rec"].notna() & (c["rec"] <= 365 - (c["real_cutoff"] - c["cutoff"]).dt.days)].copy()
    fut = m[(m["order_date"] >= m["scoring_date"]) & (m["order_date"] < m["win_end"])]
    c["target"] = c["customer_id"].isin(fut["customer_id"]).astype(float)
    h = orders.merge(c[["customer_id", "cutoff"]], on="customer_id")
    h = h[h["order_date"] <= h["cutoff"]][orders.columns].copy()
    c["src_id"], c["customer_id"] = c["customer_id"], c["customer_id"] + k * ID_OFF
    h["customer_id"] += k * ID_OFF
    c["part"], c["shift"] = "aug", s
    c["slice"] = c["scoring_date"].dt.strftime("%Y-%m-%d")
    c["win_len"] = (c["win_end"] - c["scoring_date"]).dt.days
    aug_c.append(c[["customer_id", "scoring_date", "target", "part", "slice", "cutoff", "win_len", "src_id", "shift"]])
    aug_o.append(h)
    print(f"сдвиг {s} мес: псевдоклиентов {len(c)}, доля target {c['target'].mean():.3f}, строк заказов {len(h)}")
cust = pd.concat([real.assign(src_id=real["customer_id"], shift=0)] + aug_c, ignore_index=True)
orders = pd.concat([orders] + aug_o, ignore_index=True)
assert cust["customer_id"].is_unique
print("доля target в настоящем train:", round(train["target"].mean(), 3), "| клиентов всего:", len(cust))

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


feats = build_features(lines, cust.set_index("customer_id")["win_len"])
data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
assert data["recency"].notna().all() and len(data) == len(cust)
data["top_category2"] = data["top_category2"].astype("category")
data["top_brand"] = data["top_brand"].astype(object).where(data["top_brand"].notna(), "none").astype(str)

## Признаки по отдельным заказам, интервалам и календарю

In [ ]:
cid = "customer_id"
info = cust.set_index(cid)
LO = pd.Series(float(GAP_DAYS), index=info.index)
HI = (GAP_DAYS + info["win_len"]).astype(float)
EST = ["mean", "median", "min", "max", "q25", "q75", "last", "last2", "last3", "ewm", "trim"]
HIST = {"вся": 10_000, "год": 365, "полгода": 182}


def month_idx(s):
    return s.dt.year * 12 + s.dt.month


def estimators(E):
    E = E.sort_values([cid, "t"]).reset_index(drop=True)
    g = E.groupby(cid)["t"]
    E["gap"] = g.diff()
    E["rk"] = E.groupby(cid).cumcount()
    gp = E[E["gap"].notna()]
    gg = gp.groupby(cid)["gap"]
    out = pd.DataFrame({"r": g.min(), "n": g.size()})
    out["mean"] = (g.max() - g.min()) / (out["n"] - 1).replace(0, np.nan)
    out["median"] = gg.median()
    out["min"] = gg.min()
    out["max"] = gg.max()
    out["q25"] = gg.quantile(0.25)
    out["q75"] = gg.quantile(0.75)
    out["last"] = gp[gp["rk"] == 1].set_index(cid)["gap"]
    out["last2"] = gp[gp["rk"] <= 2].groupby(cid)["gap"].mean()
    out["last3"] = gp[gp["rk"] <= 3].groupby(cid)["gap"].mean()
    w = 0.7 ** (gp["rk"] - 1)
    out["ewm"] = (gp["gap"] * w).groupby(gp[cid]).sum() / w.groupby(gp[cid]).sum()
    out["trim"] = (gg.sum() - gg.max()) / (gg.count() - 1).replace(0, np.nan)
    return out


def unit_feats(E, prefix):
    # «ожидаемый момент следующего заказа» по разным оценкам периода; E — (customer_id, d), d — дней до отсечения
    out = {}
    E = E.assign(t=E["d"].astype(float))
    for hname, H in HIST.items():
        est = estimators(E[E["d"] <= H][[cid, "t"]]).reindex(info.index)
        r = est["r"]
        key = f"{prefix}|{hname}"
        out[f"{key}|n"] = est["n"].fillna(0)
        out[f"{key}|r"] = r
        for e in EST:
            P = est[e].where(est[e] > 0)
            tn, tm = P - r, P - np.mod(r, P)
            pre = f"{key}|{e}"
            out[f"{pre}|t"] = tn
            out[f"{pre}|dist"] = (LO - tn).clip(lower=0) + (tn - HI).clip(lower=0)
            out[f"{pre}|in"] = ((tn > LO) & (tn <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|tm_in"] = ((tm > LO) & (tm <= HI)).astype("float32").where(P.notna())
            out[f"{pre}|k_in"] = np.floor((r + HI) / P) - np.floor((r + LO) / P)
    return out


o = (
    lines.groupby([cid, "order_id"], sort=False)
    .agg(d=("d", "first"), date=("order_date", "first"), n_items=("paid_price", "size"))
    .reset_index()
    .sort_values([cid, "d", "order_id"])
    .reset_index(drop=True)
)
new = unit_feats(o[[cid, "d"]], "заказ")

# структура интервалов между заказами
o["gap"] = o.groupby(cid)["d"].diff()
gpo = o[o["gap"].notna()]
for name, cond in [("0", gpo["gap"] == 0), ("le1", gpo["gap"] <= 1), ("le3", gpo["gap"] <= 3), ("le7", gpo["gap"] <= 7), ("le14", gpo["gap"] <= 14),
                   ("le30", gpo["gap"] <= 30), ("gt60", gpo["gap"] > 60), ("gt90", gpo["gap"] > 90), ("gt180", gpo["gap"] > 180)]:
    cf = cond.astype(float)
    new[f"интервалы|доля {name}"] = cf.groupby(gpo[cid]).mean()
    new[f"интервалы|доля {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).mean()
    new[f"интервалы|число {name} за год"] = cf.where(gpo["d"] <= 365).groupby(gpo[cid]).sum()
dayc = o.groupby([cid, "d"]).agg(k=("order_id", "size"), items=("n_items", "sum")).reset_index()
for w in (90, 365):
    new[f"интервалы|дней с 2+ заказами за {w}"] = ((dayc["k"] >= 2) & (dayc["d"] <= w)).groupby(dayc[cid]).sum()
last_day = dayc.sort_values([cid, "d"]).drop_duplicates(cid).set_index(cid)
new["интервалы|заказов в последний день"] = last_day["k"]
new["интервалы|позиций в последний день"] = last_day["items"]
new["интервалы|давность последнего дня с 2+ заказами"] = dayc[dayc["k"] >= 2].groupby(cid)["d"].min()
r_o = o.groupby(cid)["d"].min()
qs = gpo.groupby(cid)["gap"].quantile([0.25, 0.5, 0.75, 0.9]).unstack()
for q in qs.columns:
    new[f"интервалы|давность / квантиль {q}"] = r_o / qs[q].replace(0, np.nan)
    new[f"интервалы|давность - квантиль {q}"] = r_o - qs[q]
new["интервалы|разброс"] = gpo.groupby(cid)["gap"].std() / gpo.groupby(cid)["gap"].mean().replace(0, np.nan)
pos = gpo[gpo["gap"] > 0].groupby(cid)["gap"]
new["интервалы|средний ненулевой"] = pos.mean()
new["интервалы|давность - средний ненулевой"] = r_o - pos.mean()

# заказы и позиции по непересекающимся интервалам до отсечения
BINS = [0, 7, 14, 30, 60, 90, 120, 180, 270, 365, 450, 540, 733]
labels = [f"{BINS[i]}-{BINS[i + 1]}" for i in range(len(BINS) - 1)]
t_o = pd.crosstab(o[cid], pd.cut(o["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
t_l = pd.crosstab(lines[cid], pd.cut(lines["d"], BINS, right=False, labels=labels)).reindex(columns=labels, fill_value=0)
for lab in labels:
    new[f"по интервалам|заказов {lab}"] = t_o[lab]
    new[f"по интервалам|позиций {lab}"] = t_l[lab]
    new[f"по интервалам|доля заказов {lab}"] = t_o[lab] / t_o.sum(axis=1)

# календарь
mi = month_idx(o["date"])
rel = o[cid].map(month_idx(info["scoring_date"])) - mi
last_o = o.drop_duplicates(cid).set_index(cid)
new["календарь|день месяца последнего заказа"] = last_o["date"].dt.day
new["календарь|день недели последнего заказа"] = last_o["date"].dt.dayofweek
new["календарь|месяц последнего заказа"] = last_o["date"].dt.month
new["календарь|доля заказов 1–10 числа"] = (o["date"].dt.day <= 10).groupby(o[cid]).mean()
new["календарь|доля заказов 21+ числа"] = (o["date"].dt.day >= 21).groupby(o[cid]).mean()
new["календарь|заказов в месяцы окна год назад"] = rel.between(10, 12).groupby(o[cid]).sum()
new["календарь|заказов в месяцы окна два года назад"] = rel.between(22, 24).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца перед окном год назад"] = rel.between(13, 15).groupby(o[cid]).sum()
new["календарь|заказов в 3 месяца после окна год назад"] = rel.between(7, 9).groupby(o[cid]).sum()
new["календарь|активных месяцев"] = mi.groupby(o[cid]).nunique()
new["календарь|активных месяцев за год"] = mi.where(rel <= 12).groupby(o[cid]).nunique()

NEW = pd.DataFrame(new).reindex(info.index).astype("float32")
NAMES = {f"n{i}": c for i, c in enumerate(NEW.columns)}       # короткие имена для моделей -> понятные имена
NEW.columns = list(NAMES)
data = data.join(NEW, on=cid)
BASE = [c for c in feats.columns if c != "top_brand"]
print(f"новых признаков: {NEW.shape[1]} | {(time.time() - T0) / 60:.1f} мин")

## Отбор новых признаков

In [ ]:
# Новый признак остаётся, если его важность выше 90-го процентиля важности на перемешанном таргете
trd = data[data["part"].eq("train")]
Xs, ys = trd[BASE + list(NAMES)], trd["target"].astype(int).to_numpy()
sel = dict(objective="binary", learning_rate=0.05, n_estimators=300, num_leaves=31, min_child_samples=100,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, n_jobs=-1, verbose=-1)


def gain(target, seed):
    m = lgb.LGBMClassifier(**sel, random_state=seed).fit(Xs, target)
    return pd.Series(m.booster_.feature_importance("gain"), index=Xs.columns)


rng = np.random.default_rng(SEED)
real = pd.concat([gain(ys, s) for s in range(3)], axis=1).mean(axis=1)
null = pd.concat([gain(rng.permutation(ys), s) for s in range(6)], axis=1)
ratio = real / (null.quantile(0.9, axis=1) + 1e-9)
KEEP_NEW = [c for c in NAMES if ratio[c] > 1.0]
FEATURES = BASE + KEEP_NEW
print(f"новых признаков выше шума: {len(KEEP_NEW)} из {len(NAMES)} | всего признаков: {len(FEATURES)} | {(time.time() - T0) / 60:.1f} мин")
print("Самые важные из новых:")
display((real[KEEP_NEW] / real.sum()).sort_values(ascending=False).head(20).rename(index=NAMES).to_frame("доля важности"))

## Банк признаков

In [ ]:
IDX = info.index
L = lines
OT = (L.groupby([cid, "order_id"]).agg(d=("d", "first"), s=("paid_price", "sum"), n=("paid_price", "size"),
                                       disc=("discount_percent", "mean"), loy=("loyalty_percent", "max"))
      .reset_index().sort_values([cid, "d", "order_id"]).reset_index(drop=True))
OT["k"] = OT.groupby(cid).cumcount()                 # 0 — самый свежий заказ
OT["gap"] = OT.groupby(cid)["d"].diff()              # пауза до соседнего более свежего заказа
R = OT.groupby(cid)["d"].min().reindex(IDX)
BANK = {}


def put(group, name, s):
    BANK.setdefault(group, {})[name] = pd.Series(s).reindex(IDX)


def ratio(a, b):
    return a / b.replace(0, np.nan)


# окна по времени и тренды между соседними окнами
W = [30, 60, 90, 120, 180, 270, 365, 540, 730]
cnt, summ = {}, {}
for w in W:
    m = OT["d"] < w
    cnt[w] = m.groupby(OT[cid]).sum().reindex(IDX).fillna(0)
    summ[w] = OT["s"].where(m, 0).groupby(OT[cid]).sum().reindex(IDX).fillna(0)
    put("окна", f"заказов <{w}", cnt[w])
    put("окна", f"сумма <{w}", summ[w])
    put("окна", f"позиций <{w}", OT["n"].where(m, 0).groupby(OT[cid]).sum())
    put("окна", f"скидка <{w}", OT["disc"].where(m).groupby(OT[cid]).mean())
for a, b in zip(W[:-1], W[1:]):
    put("окна", f"тренд заказов {a}/{b}", ratio(cnt[a], cnt[b] - cnt[a] + 1))
    put("окна", f"тренд суммы {a}/{b}", ratio(summ[a], summ[b] - summ[a] + 1))
    put("окна", f"доля заказов {a} из {b}", ratio(cnt[a], cnt[b]))

# последние k заказов
for k in [2, 3, 5, 10]:
    sub = OT[OT["k"] < k]
    g = sub.groupby(cid)
    c = g.size()
    span = g["d"].max() - g["d"].min()
    put("последние k", f"сумма ср {k}", g["s"].mean())
    put("последние k", f"позиций ср {k}", g["n"].mean())
    put("последние k", f"скидка ср {k}", g["disc"].mean())
    put("последние k", f"размах {k}", span.where(c == k))
    put("последние k", f"пауза ср {k}", (span / (c - 1).replace(0, np.nan)).where(c == k))
    put("последние k", f"давность {k}-го", g["d"].max().where(c == k))
    put("последние k", f"давность / пауза {k}", ratio(R, (span / (c - 1).replace(0, np.nan)).where(c == k).reindex(IDX)))

# статистики пауз по окнам
gp = OT[OT["gap"].notna()]
for H in [90, 180, 365, 730]:
    g = gp[gp["d"] < H].groupby(cid)["gap"]
    mean, med = g.mean().reindex(IDX), g.median().reindex(IDX)
    put("паузы", f"ср <{H}", mean)
    put("паузы", f"медиана <{H}", med)
    put("паузы", f"разброс <{H}", g.std())
    put("паузы", f"мин <{H}", g.min())
    put("паузы", f"макс <{H}", g.max())
    put("паузы", f"асимметрия <{H}", g.skew())
    for q in [7, 30, 90]:
        put("паузы", f"доля ≤{q} <{H}", (gp["gap"] <= q).where(gp["d"] < H).groupby(gp[cid]).mean())
    put("паузы", f"давность - ср <{H}", R - mean)
    put("паузы", f"давность / медиана <{H}", ratio(R, med))
    put("паузы", f"макс - давность <{H}", g.max().reindex(IDX) - R)

# повторы товаров, брендов и подкатегорий
L2 = L[[cid, "d", "product_id", "brand", "category2", "category"]].copy()
L2["last"] = L2["d"].eq(L2[cid].map(R))
for key in ["product_id", "brand", "category2", "category"]:
    u = L2.dropna(subset=[key]).drop_duplicates([cid, "d", key])
    days_per = u.groupby([cid, key]).size()
    put("повторы", f"{key}: доля повторных", (days_per >= 2).groupby(level=0).mean())
    put("повторы", f"{key}: макс дней", days_per.groupby(level=0).max())
    before = u[~u["last"]][[cid, key]].drop_duplicates().assign(seen=1)
    lastk = u[u["last"]][[cid, key]].drop_duplicates().merge(before, on=[cid, key], how="left")
    put("повторы", f"{key}: последний заказ уже покупал", lastk["seen"].fillna(0).groupby(lastk[cid]).mean())

# денежные тренды
for a, b in [(90, 180), (180, 365), (365, 730)]:
    put("деньги", f"сумма {a} / {a}-{b}", ratio(summ[a], summ[b] - summ[a]))
put("деньги", "средний чек 365 / 730", ratio(ratio(summ[365], cnt[365]), ratio(summ[730], cnt[730])))
put("деньги", "чек последнего / средний", ratio(OT[OT["k"] == 0].set_index(cid)["s"].reindex(IDX), OT.groupby(cid)["s"].mean().reindex(IDX)))
put("деньги", "лояльность сейчас - год назад", OT[OT["k"] == 0].set_index(cid)["loy"].reindex(IDX)
    - OT[OT["d"] >= 300].groupby(cid)["loy"].first().reindex(IDX))
put("деньги", "скидка 90 - скидка 365", OT["disc"].where(OT["d"] < 90).groupby(OT[cid]).mean().reindex(IDX)
    - OT["disc"].where(OT["d"] < 365).groupby(OT[cid]).mean().reindex(IDX))

# давность и частота по каждой категории
for c in L["category"].dropna().unique():
    s = L[L["category"] == c]
    dd = s.drop_duplicates([cid, "d"])
    put("категории", f"{c}: давность", dd.groupby(cid)["d"].min())
    put("категории", f"{c}: дней <365", (dd["d"] < 365).groupby(dd[cid]).sum())
    put("категории", f"{c}: доля позиций <180", ratio((s["d"] < 180).groupby(s[cid]).sum().reindex(IDX),
                                                       (L["d"] < 180).groupby(L[cid]).sum().reindex(IDX)))

# бренды
b365 = L[L["d"] < 365]
put("бренды", "брендов <365", b365.groupby(cid)["brand"].nunique())
put("бренды", "брендов <90", L[L["d"] < 90].groupby(cid)["brand"].nunique())
put("бренды", "брендов на заказ", ratio(L.groupby(cid)["brand"].nunique().reindex(IDX), OT.groupby(cid).size().reindex(IDX)))
topb = L.groupby([cid, "brand"]).size()
put("бренды", "доля главного бренда", topb.groupby(level=0).max() / topb.groupby(level=0).sum())
put("бренды", "подкатегорий <365", b365.groupby(cid)["category2"].nunique())

GROUPS, BANK_NAMES = {}, {}
for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
print("групп:", len(GROUPS), "| признаков в банке:", len(BANK_NAMES), {g: len(c) for g, c in GROUPS.items()}, f"| {(time.time() - T0) / 60:.1f} мин")

## Паузы внутри категорий

In [ ]:
BANK = {}
for col in ["category", "category2"]:
    dd = L[[cid, col, "d"]].dropna().drop_duplicates().sort_values([cid, col, "d"])
    dd["gap"] = dd.groupby([cid, col])["d"].diff()
    st = dd.groupby([cid, col]).agg(last=("d", "min"), n=("d", "size"), med=("gap", "median"), mx=("gap", "max")).reset_index()
    st = st[st["n"] >= 2]
    st["r_med"] = st["last"] / st["med"].replace(0, np.nan)
    st["med_r"] = st["med"] - st["last"]
    st["mx_r"] = st["mx"] - st["last"]
    g = st.groupby(cid)
    put("паузы в категориях", f"{col}: мин давность / медиана", g["r_med"].min())
    put("паузы в категориях", f"{col}: ср давность / медиана", g["r_med"].mean())
    put("паузы в категориях", f"{col}: медиана - давность, макс", g["med_r"].max())
    put("паузы в категориях", f"{col}: медиана - давность, мин |.|", st.assign(a=st["med_r"].abs()).groupby(cid)["a"].min())
    put("паузы в категориях", f"{col}: макс пауза - давность, макс", g["mx_r"].max())
    put("паузы в категориях", f"{col}: серий 3+ дней", (st["n"] >= 3).groupby(st[cid]).sum())
    top = st.sort_values([cid, "n", "last"], ascending=[True, False, True]).drop_duplicates(cid).set_index(cid)
    put("паузы в категориях", f"{col} главная: медиана", top["med"])
    put("паузы в категориях", f"{col} главная: давность", top["last"])
    put("паузы в категориях", f"{col} главная: давность / медиана", top["r_med"])

for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
FEATURES_V2 = FEATURES
FEATURES = FEATURES_V2 + GROUPS["паузы"] + GROUPS["категории"] + GROUPS["паузы в категориях"]
print("признаков v2:", len(FEATURES_V2), "| с банком:", len(FEATURES), f"| {(time.time() - T0) / 60:.1f} мин")

## Паузы по каждой категории и окно по категориям

In [ ]:
FEATURES_V4 = FEATURES
BANK = {}
HI_ = GAP_DAYS + info["win_len"].astype(float)       # целевое окно в днях после отсечения: (GAP_DAYS, HI_]


def cycles(col, src=None):
    # по каждой паре (клиент, значение col): давность последней покупки, первой, число дней, медиана и максимум паузы
    src = L if src is None else src
    dd = src[[cid, col, "d"]].dropna().drop_duplicates().sort_values([cid, col, "d"])
    dd["gap"] = dd.groupby([cid, col])["d"].diff()
    st = dd.groupby([cid, col]).agg(last=("d", "min"), first=("d", "max"), n=("d", "size"), med=("gap", "median"),
                                     mx=("gap", "max")).reset_index()
    st = st[st["n"] >= 2].copy()
    st["r_med"] = st["last"] / st["med"].replace(0, np.nan)
    st["t_med"] = st["med"] - st["last"]                                  # ожидаемая следующая покупка по медиане
    st["t_mean"] = (st["first"] - st["last"]) / (st["n"] - 1) - st["last"]  # то же по средней паузе
    return st


# паузы отдельно по каждой категории и полу товара
for col in ["category", "gender_product"]:
    st = cycles(col)
    for c in L[col].dropna().unique():
        s = st[st[col] == c].set_index(cid)
        put("паузы по каждой категории", f"{col} {c}: медиана", s["med"])
        put("паузы по каждой категории", f"{col} {c}: давность / медиана", s["r_med"])
        put("паузы по каждой категории", f"{col} {c}: медиана - давность", s["t_med"])
        put("паузы по каждой категории", f"{col} {c}: средняя - давность", s["t_mean"])
        put("паузы по каждой категории", f"{col} {c}: макс пауза", s["mx"])

# попадает ли ожидаемая покупка по категориям в целевое окно
for col in ["category", "category2", "brand"]:
    st = cycles(col)
    hi = st[cid].map(HI_)
    mid = (GAP_DAYS + hi) / 2
    for t in ["t_med", "t_mean"]:
        inn = ((st[t] > GAP_DAYS) & (st[t] <= hi)).astype(float)
        dist = (GAP_DAYS - st[t]).clip(lower=0) + (st[t] - hi).clip(lower=0)
        put("окно по категориям", f"{col} {t}: в окне", inn.groupby(st[cid]).sum())
        put("окно по категориям", f"{col} {t}: доля в окне", inn.groupby(st[cid]).mean())
        put("окно по категориям", f"{col} {t}: доля в окне, вес по дням", (inn * st["n"]).groupby(st[cid]).sum() / st["n"].groupby(st[cid]).sum())
        put("окно по категориям", f"{col} {t}: мин расстояние", dist.groupby(st[cid]).min())
        put("окно по категориям", f"{col} {t}: мин |t - середина окна|", (st[t] - mid).abs().groupby(st[cid]).min())
    put("окно по категориям", f"{col}: t_mean в (35, 85]", st["t_mean"].between(35, 85, inclusive="right").groupby(st[cid]).sum())

for gname, cols in BANK.items():
    df = pd.DataFrame(cols).replace([np.inf, -np.inf], np.nan).astype("float32")
    short = [f"b_{len(BANK_NAMES) + i}" for i in range(df.shape[1])]
    BANK_NAMES.update(dict(zip(short, df.columns)))
    df.columns = short
    data = data.join(df, on=cid)
    GROUPS[gname] = short
FEATURES = FEATURES_V4 + GROUPS["паузы по каждой категории"] + GROUPS["окно по категориям"]
print("признаков v4:", len(FEATURES_V4), "| v5:", len(FEATURES), f"| {(time.time() - T0) / 60:.1f} мин")

## Псевдосрезы

In [ ]:
tr = data[data["part"].eq("train")].reset_index(drop=True)
au = data[data["part"].eq("aug")].reset_index(drop=True)
te = data[data["part"].eq("test")].set_index(cid).loc[sample[cid]].reset_index()
y, sl = tr["target"].astype(int).to_numpy(), tr["slice"].to_numpy()
ya = au["target"].astype(int).to_numpy()
print("Псевдосрезы по сдвигу и дате оценки:")
display(au.groupby(["shift", "slice"]).agg(клиентов=("target", "size"), доля_target=("target", "mean")))
cmp = pd.DataFrame({
    "настоящие": tr.groupby(pd.cut(tr["recency"], [-1, 30, 60, 120, 243, 365]), observed=True)["target"].mean(),
    "сдвиг 4": au[au["shift"] == 4].groupby(pd.cut(au.loc[au["shift"] == 4, "recency"], [-1, 30, 60, 120, 243, 365]), observed=True)["target"].mean(),
    "сдвиг 8": au[au["shift"] == 8].groupby(pd.cut(au.loc[au["shift"] == 8, "recency"], [-1, 30, 60, 120, 243, 365]), observed=True)["target"].mean(),
})
print("Доля target по давности: если отбор не смещён, псевдосрезы похожи на настоящие (разница — сезон и длина истории)")
display(cmp)

## Перенос на март

In [ ]:
P = dict(objective="binary", learning_rate=0.05, n_estimators=3000, subsample_freq=1, num_leaves=47, min_child_samples=85,
         colsample_bytree=0.6, subsample=0.89, reg_lambda=0.66, min_split_gain=0.43, n_jobs=-1, verbose=-1)
COLS = FEATURES
MAR = np.where(sl == "2022-03-01")[0]
JF = np.where(sl != "2022-03-01")[0]
yb = y[MAR]
BOOT = [np.random.default_rng(i).integers(0, len(MAR), len(MAR)) for i in range(300)]
SEEDS = [0, 1, 2]


def fit_predict(Xa, ya_, wa, Xe, ye, Xp, seed):
    m = lgb.LGBMClassifier(**P, random_state=seed).fit(Xa, ya_, sample_weight=wa, eval_set=[(Xe, ye)],
                                                     callbacks=[lgb.early_stopping(100, verbose=False)])
    return m.predict_proba(Xp)[:, 1]


def march(aug_mask=None, w_aug=1.0):
    # обучение на январе и феврале (+ псевдосрезы), ранняя остановка по 10% настоящих строк, прогноз на март; среднее по зёрнам
    pred = np.zeros(len(MAR))
    for s in SEEDS:
        perm = np.random.default_rng(SEED + s).permutation(JF)
        aa, es = perm[: int(len(JF) * 0.9)], perm[int(len(JF) * 0.9):]
        Xa, yy, ww = tr[COLS].iloc[aa], y[aa], np.ones(len(aa))
        if aug_mask is not None:
            Xa = pd.concat([Xa, au.loc[aug_mask, COLS]])
            yy = np.concatenate([yy, ya[aug_mask]])
            ww = np.concatenate([ww, np.full(int(aug_mask.sum()), w_aug)])
        pred += fit_predict(Xa, yy, ww, tr[COLS].iloc[es], y[es], tr[COLS].iloc[MAR], SEED + s) / len(SEEDS)
    return pred


def paired(p, q):
    # разница AP@20% на марте и её шум по бутстрепу клиентов марта (одни и те же выборки для обоих прогнозов)
    d = np.array([fast_ap(yb[i], p[i]) - fast_ap(yb[i], q[i]) for i in BOOT])
    return fast_ap(yb, p) - fast_ap(yb, q), d.std(ddof=1)


# псевдосрезы, чьё целевое окно закончилось до начала мартовского окна (1 марта 2022) — у всех сдвигов так и есть
assert (pd.to_datetime(au["scoring_date"]) + pd.DateOffset(months=TARGET_MONTHS) <= pd.Timestamp("2022-03-01")).all()
s4, s8 = au["shift"].eq(4).to_numpy(), au["shift"].eq(8).to_numpy()
variants = {"без псевдосрезов": (None, 1.0), "+ сдвиг 4": (s4, 1.0), "+ сдвиг 4 и 8": (s4 | s8, 1.0), "+ сдвиг 4 и 8, вес 0.5": (s4 | s8, 0.5)}
PRED, rows = {}, []
for name, (mask, w) in variants.items():
    t0 = time.time()
    PRED[name] = march(mask, w)
    d, se = paired(PRED[name], PRED["без псевдосрезов"]) if name != "без псевдосрезов" else (np.nan, np.nan)
    rows.append({"вариант": name, "строк обучения": len(JF) * 0.9 + (0 if mask is None else mask.sum()),
                 "AP@20% март": fast_ap(yb, PRED[name]), "прирост": d, "шум": se, "прирост / шум": d / se,
                 "AUC 1": roc_auc_score(yb[tr["n_orders"].iloc[MAR].eq(1).to_numpy()], PRED[name][tr["n_orders"].iloc[MAR].eq(1).to_numpy()]),
                 "AUC 4+": roc_auc_score(yb[tr["n_orders"].iloc[MAR].ge(4).to_numpy()], PRED[name][tr["n_orders"].iloc[MAR].ge(4).to_numpy()]),
                 "минут": round((time.time() - t0) / 60, 1)})
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rows[-1].items()}, flush=True)
march_res = pd.DataFrame(rows).set_index("вариант")
display(march_res)

## Кросс-валидация

In [ ]:
# 5 фолдов по настоящим строкам; в обучение каждого фолда добавляются все псевдосрезы. Их таргет целиком лежит в видимой
# истории клиентов, то есть уже содержится в признаках; на тесте псевдосрезы тестовых клиентов тоже есть в обучении.
best = max([k for k in variants if k != "без псевдосрезов"], key=lambda k: march_res.loc[k, "AP@20% март"])
mask_b, w_b = variants[best]
strata = tr["slice"] + "_" + tr["target"].astype(int).astype(str)
SPLITS = [list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + s).split(tr, strata)) for s in range(2)]


def ap_rows(rows, pred):
    return float(np.mean([fast_ap(y[rows][sl[rows] == s], pred[sl[rows] == s]) for s in np.unique(sl[rows])]))


def run(mask, w):
    per_fold = []
    for split in SPLITS:
        for a, b in split:
            Xa, yy, ww = tr[COLS].iloc[a], y[a], np.ones(len(a))
            if mask is not None:
                Xa, yy = pd.concat([Xa, au.loc[mask, COLS]]), np.concatenate([yy, ya[mask]])
                ww = np.concatenate([ww, np.full(int(mask.sum()), w)])
            p = fit_predict(Xa, yy, ww, tr[COLS].iloc[b], y[b], tr[COLS].iloc[b], SEED)
            per_fold.append(ap_rows(b, p))
    return np.array(per_fold)


cv_base = run(None, 1.0)
cv_aug = run(mask_b, w_b)
diff = cv_aug - cv_base
print(f"CV без псевдосрезов {cv_base.mean():.4f} | {best}: {cv_aug.mean():.4f} | прирост {diff.mean():+.4f} ± {diff.std(ddof=1) / np.sqrt(len(diff)):.4f}"
      f" | {(time.time() - T0) / 60:.1f} мин")

## Пробный сабмит

In [ ]:
test = np.zeros(len(te))
for a, b in SPLITS[0]:
    Xa, yy, ww = tr[COLS].iloc[a], y[a], np.ones(len(a))
    if mask_b is not None:
        Xa, yy = pd.concat([Xa, au.loc[mask_b, COLS]]), np.concatenate([yy, ya[mask_b]])
        ww = np.concatenate([ww, np.full(int(mask_b.sum()), w_b)])
    test += fit_predict(Xa, yy, ww, tr[COLS].iloc[b], y[b], te[COLS], SEED) / N_FOLDS
sub = pd.DataFrame({cid: te[cid].to_numpy(), "score": pd.Series(test).rank(pct=True).to_numpy()})
assert sub[cid].tolist() == sample[cid].tolist()
sub.to_csv(OUT_DIR / "submission_shift_lgb.csv", index=False)
json.dump({"best": best, "shifts": SHIFTS, "march": march_res.reset_index().to_dict("records"), "cv_base": float(cv_base.mean()),
           "cv_aug": float(cv_aug.mean())}, open(OUT_DIR / "shift.json", "w"), ensure_ascii=False, indent=1, default=float)
print(f"Сохранено: submission_shift_lgb.csv ({best}). Всего времени: {(time.time() - T0) / 60:.1f} мин")